# CatBoost Classifier

In [1]:
# imports
%pip install pandas scikit-learn catboost

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

Note: you may need to restart the kernel to use updated packages.


## Data preparation

In [2]:
df_challenger = pd.read_csv('../datasets/challenger_dataset.csv')

df_challenger.head()

,timestamp,BLUE.HEXTECH_DRAGON.kills,BLUE.WATER_DRAGON.kills,BLUE.FIRE_DRAGON.kills,BLUE.EARTH_DRAGON.kills,BLUE.AIR_DRAGON.kills,BLUE.CHEMTECH_DRAGON.kills,BLUE.ELDER_DRAGON.kills,BLUE.RIFTHERALD.kills,BLUE.BARON_NASHOR.kills,...,soul_Mountain,soul_Ocean,BLUE.minionRatio,RED.minionRatio,BLUE.totalGoldRatio,RED.totalGoldRatio,BLUE.levelRatio,RED.levelRatio,BLUE.kda,RED.kda
0,0,0,0,0,0,0,0,0,0,0,...,0,0,NaN,NaN,inf,inf,inf,inf,NaN,NaN
1,60017,0,0,0,0,0,0,0,0,0,...,0,0,0.000000,0.000000,0.041655,0.041655,0.000083,0.000083,NaN,NaN
2,120046,0,0,0,0,0,0,0,0,0,...,0,0,0.000067,0.000092,0.024016,0.027648,0.000050,0.000050,0.000000,inf
3,180116,0,0,0,0,0,0,0,0,0,...,0,0,0.000333,0.000272,0.024978,0.026333,0.000072,0.000072,0.000000,inf
4,240131,0,0,0,0,0,0,0,0,0,...,0,0,0.000362,0.000346,0.026027,0.028230,0.000067,0.000075,0.666667,6.0


In [3]:
X = df_challenger.drop('winner', axis=1)
y = df_challenger['winner']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Model Initialization

In [5]:
# Initialize the CatBoost model
model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    loss_function='Logloss',
    eval_metric='Accuracy',
    verbose=False
)

# Train the model
model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    early_stopping_rounds=10
)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

Model Accuracy: 0.7308


## Hyperparameter Optimization

In [ ]:
# Define the parameter distributions for RandomizedSearchCV
param_distributions = {
    'iterations': randint(100, 500),
    'learning_rate': uniform(0.01, 0.3),
    'depth': randint(4, 10),
    'l2_leaf_reg': randint(1, 10),
    'bootstrap_type': ['Bayesian', 'Bernoulli', 'MVS'],
    'random_seed': [42]
}

# Initialize the base model
base_model = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='Accuracy',
    verbose=False
)

# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_distributions,
    n_iter=50,  # Number of parameter settings sampled
    cv=5,       # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,  # Use all available cores
    verbose=10,
    random_state=42
)

# Perform random search
random_search.fit(X_train, y_train)

# Print best parameters
print("\nBest parameters found:")
print(random_search.best_params_)


Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 1/5; 1/50] START bootstrap_type=MVS, depth=7, iterations=448, l2_leaf_reg=8, learning_rate=0.189597545259111, random_seed=42
[CV 4/5; 1/50] START bootstrap_type=MVS, depth=7, iterations=448, l2_leaf_reg=8, learning_rate=0.189597545259111, random_seed=42
[CV 1/5; 2/50] START bootstrap_type=MVS, depth=5, iterations=314, l2_leaf_reg=8, learning_rate=0.11011258334170654, random_seed=42
[CV 5/5; 1/50] START bootstrap_type=MVS, depth=7, iterations=448, l2_leaf_reg=8, learning_rate=0.189597545259111, random_seed=42
[CV 2/5; 1/50] START bootstrap_type=MVS, depth=7, iterations=448, l2_leaf_reg=8, learning_rate=0.189597545259111, random_seed=42
[CV 3/5; 1/50] START bootstrap_type=MVS, depth=7, iterations=448, l2_leaf_reg=8, learning_rate=0.189597545259111, random_seed=42
[CV 2/5; 2/50] START bootstrap_type=MVS, depth=5, iterations=314, l2_leaf_reg=8, learning_rate=0.11011258334170654, random_seed=42
[CV 3/5; 2/50] START bootstrap_

In [8]:
# Print best score
print(f"\nBest cross-validation accuracy: {random_search.best_score_:.4f}")

# Get the best model
best_model = random_search.best_estimator_

# Evaluate the best model on test set
y_pred_best = best_model.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)
print(f"\nTest set accuracy with best parameters: {accuracy_best:.4f}")


Best cross-validation accuracy: 0.7625

Test set accuracy with best parameters: 0.7656
